In [1]:
import pandas as pd
import numpy as np

print("Загрузка данных...")

df_interactions_train = pd.read_csv('interactions_train.csv')
df_users = pd.read_csv('users.csv')
df_items = pd.read_csv('items.csv')
df_interactions_test = pd.read_csv('interactions_private_test.csv')

df_interactions_train['target'] = (df_interactions_train['watched_pct'] > 50.0).astype(int)

df_interactions_train = df_interactions_train.drop(columns=['watched_pct'])

print("Первые 5 строк обучающей выборки:")
display(df_interactions_train.head())

Загрузка данных...
Первые 5 строк обучающей выборки:


,user_id,item_id,last_watch_dt,total_dur,target
0,3,10139,2021-04-25,103.0,0
1,3,7204,2021-07-17,28.0,0
2,3,12928,2021-08-17,845.0,0
3,3,3897,2021-08-17,901.0,0
4,4,811,2021-06-12,6191.0,1


In [2]:
df_train = df_interactions_train.merge(df_users, on='user_id', how='left')

df_train = df_train.merge(df_items, on='item_id', how='left')

print(f"Объединенная обучающая выборка готова. Новый размер: {df_train.shape}")
print("Первые 5 строк:")
display(df_train.head(2))

Объединенная обучающая выборка готова. Новый размер: (922967, 21)
Первые 5 строк:


,user_id,item_id,last_watch_dt,total_dur,target,age,income,sex,kids_flg,content_type,...,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,keywords
0,3,10139,2021-04-25,103.0,0,age_35_44,income_20_40,Ж,1.0,film,...,TitleOrig_008670,2016.0,"Genre_000008, Genre_000035, Genre_000007","Country_000003, Country_000047",NaN,16.0,NaN,Director_007504,"Actor_022220, Actor_006456, Actor_053457, Acto...","Keyword_000799, Keyword_021251, Keyword_009086..."
1,3,7204,2021-07-17,28.0,0,age_35_44,income_20_40,Ж,1.0,film,...,TitleOrig_008262,2023.0,Genre_000005,Country_000005,NaN,16.0,NaN,Director_000357,"Actor_008985, Actor_028967, Actor_000772, Acto...","Keyword_000512, Keyword_001401, Keyword_001140..."


In [3]:
print("Объединяем тестовые данные...")

df_test_submission = df_interactions_test[['user_id', 'item_id']].copy()

df_test = df_interactions_test.merge(df_users, on='user_id', how='left')

df_test = df_test.merge(df_items, on='item_id', how='left')

print(f"Объединенная тестовая выборка готова. Размер: {df_test.shape}")

Объединяем тестовые данные...
Объединенная тестовая выборка готова. Размер: (100782, 20)


In [4]:
def extract_date_features(df, column):
    if column not in df.columns: 
        print(f"Столбец '{column}' не найден, пропускаем обработку.")
        return df

    df[column] = pd.to_datetime(df[column], errors='coerce') 
    
    df['dt_dayofweek'] = df[column].dt.dayofweek
    df['dt_hour'] = df[column].dt.hour
    
    df = df.drop(columns=[column], errors='ignore') 
    return df

df_train = extract_date_features(df_train, 'last_watch_dt') 
df_test = extract_date_features(df_test, 'last_watch_dt')

print("Признаки из даты извлечены.")

print("\nПроверка dt_dayofweek в df_test:", 'dt_dayofweek' in df_test.columns)

Признаки из даты извлечены.

Проверка dt_dayofweek в df_test: True


In [5]:
NUMERIC_COLS = ['total_dur', 'release_year', 'for_kids', 'age_rating', 'kids_flg'] 

CATEGORICAL_COLS = ['age', 'income', 'sex', 'content_type', 'genres', 'countries', 
                    'studios', 'directors', 'actors', 'keywords']

CATEGORICAL_COLS.extend(['dt_dayofweek', 'dt_hour'])

df_train[CATEGORICAL_COLS] = df_train[CATEGORICAL_COLS].fillna('unknown')
df_test[CATEGORICAL_COLS] = df_test[CATEGORICAL_COLS].fillna('unknown') 

df_train[NUMERIC_COLS] = df_train[NUMERIC_COLS].fillna(0)
df_test[NUMERIC_COLS] = df_test[NUMERIC_COLS].fillna(0)

print("Все пропуски заполнены.")

for col in CATEGORICAL_COLS:
    df_train[col] = df_train[col].astype('category')
    if col in df_test.columns:
        df_test[col] = df_test[col].astype('category')

print("Категориальные признаки преобразованы.")

EXCLUDE_COLS = ['user_id', 'item_id', 'title', 'title_orig'] 
FEATURES = [col for col in df_train.columns if col not in EXCLUDE_COLS + ['target']]

print(f"Количество признаков для модели (X): {len(FEATURES)}")

Все пропуски заполнены.
Категориальные признаки преобразованы.
Количество признаков для модели (X): 17


In [6]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import lightgbm as lgb
import joblib 

RANDOM_SEED = 42
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,    
    'learning_rate': 0.01,    
    'random_state': RANDOM_SEED,
    'n_jobs': 4,             
    'is_unbalance': True,
    'verbose': -1
}

FEATURES = [col for col in df_train.columns if col not in ['user_id', 'item_id', 'title', 'title_orig', 'target']]
X = df_train[FEATURES]
y = df_train['target']

N_SPLITS = 5
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

f1_scores = []
final_models = []

print(f"Начало {N_SPLITS}-кратной кросс-валидации (4 ядра, n_estimators=2000)...")

for fold, (train_index, val_index) in enumerate(kf.split(X, y)):
    print(f"\n--- ФОЛД {fold+1}/{N_SPLITS} ---")

    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model_fold = lgb.LGBMClassifier(**params)
    model_fold.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='f1', 
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )

    val_predictions = model_fold.predict(X_val)
    f1 = f1_score(y_val, val_predictions)
    f1_scores.append(f1)
    
    print(f"F1-Score на валидации (ФОЛД {fold+1}): {f1:.4f}")
    
    final_models.append(model_fold)

avg_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)

print(f"Средний F1-Score по {N_SPLITS} фолдам: {avg_f1:.4f} (±{std_f1:.4f})")

best_fold_index = np.argmax(f1_scores)
model = final_models[best_fold_index] 

print(f"Финальная модель для предсказания выбрана. Фолд: {best_fold_index+1}")

joblib.dump(model, 'lgbm_tuned_best_cv_model.pkl')
print(f"Модель сохранена как lgbm_tuned_best_cv_model.pkl")

Начало 5-кратной кросс-валидации (4 ядра, n_estimators=2000)...

--- ФОЛД 1/5 ---


KeyboardInterrupt: 

In [ ]:
import pandas as pd
from IPython.display import display 

RANDOM_SEED = 42 

X_test = df_test[FEATURES] 

test_predictions = model.predict_proba(X_test)[:, 1] 

df_my_predictions = df_test[['user_id', 'item_id']].copy() 
df_my_predictions['watched_pct'] = test_predictions 

SAMPLE_SUBMISSION_FILENAME = 'sample_sub_private_test_seed_0.csv' 
try:
    df_sample = pd.read_csv(SAMPLE_SUBMISSION_FILENAME)
except FileNotFoundError:
    print(f"Критическая ошибка: Файл '{SAMPLE_SUBMISSION_FILENAME}' не найден.")
    raise SystemExit("Проверьте, что файл лежит в той же папке.")


df_submission = df_my_predictions.merge(
    df_sample[['user_id', 'item_id', 'last_watch_dt', 'total_dur']],
    on=['user_id', 'item_id'], 
    how='left'
)

FINAL_COLUMNS = ['user_id', 'item_id', 'last_watch_dt', 'total_dur', 'watched_pct']
df_submission = df_submission[FINAL_COLUMNS]

submission_path = f'submission_seed_{RANDOM_SEED}_tuned.csv'
df_submission.to_csv(submission_path, index=False)

print(f"Создан файл для отправки: {submission_path}")
print("Первые 10 строк файла:")
display(df_submission.head(10))


✅ УСПЕХ! Создан файл для отправки: submission_seed_42_tuned.csv
Первые 10 строк файла:


,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,17,846,2021-04-02,3288.0,0.996993
1,25,11126,2021-07-31,9050.0,0.999907
2,30,1372,2021-07-18,400.0,0.000170
3,33,2865,2021-07-27,1507.0,0.001076
4,33,9645,2021-08-06,732.0,0.001514
5,41,10848,2021-06-30,35811.0,0.912581
6,53,13060,2021-07-06,4735.0,0.994538
7,55,4544,2021-08-20,888.0,0.003511
8,71,8516,2021-05-22,29180.0,0.999931
9,100,6943,2021-08-13,439.0,0.001091



ФИНАЛ: Файл готов! Отправляйте!


In [ ]:
import pandas as pd

# 1. Загрузите оба файла
# Убедитесь, что имена файлов соответствуют вашим сохраненным
df_old = pd.read_csv('submission_seed_42.csv') 
df_new = pd.read_csv('submission_seed_42_tuned.csv') # Или как вы назвали файл с новой моделью

# 2. Объедините таблицы по ключам (user_id и item_id)
# Это гарантирует, что мы сравниваем одну и ту же строку
df_compare = df_old[['user_id', 'item_id', 'watched_pct']].merge(
    df_new[['user_id', 'item_id', 'watched_pct']],
    on=['user_id', 'item_id'],
    suffixes=('_old', '_new')
)

# 3. Найдите разницу в предсказаниях
df_compare['difference'] = df_compare['watched_pct_old'] != df_compare['watched_pct_new']

# 4. Рассчитайте метрики
total_predictions = len(df_compare)
changed_predictions = df_compare['difference'].sum()

# 5. Вывод результатов
print("--- Анализ Различий ---")
print(f"Общее количество предсказаний: {total_predictions}")
print(f"Количество измененных предсказаний: {changed_predictions}")
print(f"Процент измененных предсказаний: {(changed_predictions / total_predictions * 100):.3f}%")

if changed_predictions > 0:
    print("\n✅ Улучшенная модель внесла изменения. Ура!")
    # Показать 5 примеров, где предсказание изменилось
    print("\nПримеры измененных строк:")
    display(df_compare[df_compare['difference']].head())
else:
    print("🛑 Модели дали одинаковые предсказания. Убедитесь, что вы использовали разные модели.")

FileNotFoundError: [Errno 2] No such file or directory: 'submission_seed_42.csv'

In [ ]:
import joblib 

# Называем файл, чтобы легко понять, что это за модель
model_filename = 'lgbm_baseline_cv_model.pkl'

# Сохраняем вашу обученную модель
# Переменная 'model' содержит лучшую модель после кросс-валидации (см. Ячейку 7)
joblib.dump(model, model_filename)

print(f"✅ Модель успешно сохранена в файл: {model_filename}")

✅ Модель успешно сохранена в файл: lgbm_baseline_cv_model.pkl
